# Lesson 11：如何训练自己的 LLM

- **讲师**：Reza Shabani，Replit AI 负责人
- **发布日期**：2023 年 5 月 25 日
- **案例**：Ghostwriter——Replit 自研的代码补全模型（竞品：GitHub Copilot）
- **课程资源**：
  - [课程视频](https://www.youtube.com/watch?v=roEKOzxilq4&list=PL1T8fO7ArWleyIqOy37OVXsP4hFXymdOZ&index=11)
  - [课程主页](https://fullstackdeeplearning.com/llm-bootcamp/spring-2023/shabani-train-your-own/)

---

## 课程大纲

0. 为什么要训练自己的 LLM？
1. 现代 LLM 技术栈
2. 数据管道：Databricks & Hugging Face
3. 数据预处理
4. Tokenizer 训练
5. 模型训练：MosaicML & Weights & Biases
6. 测试与评估：HumanEval & Hugging Face
7. 部署：FasterTransformer、Triton Server & k8s
8. 经验教训：数据为王、评估与协作
9. 什么是优秀的 LLM 工程师？

## 0. 为什么要训练自己的 LLM？

使用 OpenAI 等第三方 API 简单快捷，但 Replit 选择自训练模型。核心原因：

| 原因 | 说明 |
|------|------|
| **定制化（Customization）** | 可以针对特定领域（如代码）深度优化，第三方通用模型难以做到 |
| **减少依赖（Reduced Dependency）** | 不受第三方服务可用性、价格策略、API 变更的制约 |
| **成本效率（Cost Efficiency）** | 大规模调用时，自建推理往往比按量付费的 API 更经济 |
| **数据隐私（Data Privacy）** | 用户代码等敏感数据无需发送给第三方 |
| **更新控制（Control over Updates）** | 自主决定何时更新模型，避免第三方模型悄然变化影响产品体验 |

### Ghostwriter：代码补全的实战案例

- Replit 的代码补全产品，直接对标 GitHub Copilot
- 核心任务：根据上下文预测并补全用户正在编写的代码
- 需要深度理解代码语义、多语言支持、低延迟响应
- 这些需求使得「自训练专用模型」优于「调用通用 API」

## 1. 现代 LLM 技术栈

Replit 的 LLM 训练栈由三个核心平台构成，分别承担不同职责：

```
原始数据（GitHub 代码、Replit 用户代码等）
        ↓
【数据处理层】Databricks + Hugging Face Datasets
        ↓
【模型训练层】MosaicML（GPU 集群 + 训练配置）
              + Weights & Biases（实验追踪）
        ↓
【部署推理层】FasterTransformer + NVIDIA Triton Server
              + Kubernetes（自动扩缩容）
```

### 各平台职责划分

| 平台 | 职责 |
|------|------|
| **Databricks** | 大规模数据管道：预处理、统计分析、数据转换 |
| **Hugging Face** | 数据集托管、预训练模型、Tokenizer、推理工具 |
| **MosaicML** | GPU 节点管理、模型训练、预配置的 LLM 训练方案 |
| **Weights & Biases** | 训练指标记录、实验对比、可视化 |
| **FasterTransformer** | 高性能推理引擎，优化 Transformer 模型推理速度 |
| **NVIDIA Triton** | 生产级模型服务框架，支持批处理和多实例 |
| **Kubernetes** | 容器编排，实现推理服务的自动扩缩容 |

> **选型原则**：每个环节选择最专业的工具，而不是追求「一体化平台」。各工具之间通过标准格式（如 Parquet、HuggingFace 格式）互通。

## 2. 数据管道：Databricks & Hugging Face

### 数据来源：The Stack

- **The Stack**：来自 GitHub Archive 的大规模代码语料库，覆盖数百种编程语言
- 关键特点：**许可证宽松（Permissively Licensed）**，可用于商业训练
- 初始处理：许可证过滤 + 近似去重（Near-deduplication）

### 为什么用 Databricks 而不是 Hugging Face 的工具？

Hugging Face 提供了数据处理工具，但 Replit 选择 Databricks 做主要转换，原因：

| 对比维度 | Hugging Face 工具 | Databricks |
|----------|------------------|------------|
| **规模** | 适合中等规模数据 | 擅长 TB 级大规模数据 |
| **数据控制** | 流程较固定 | 更灵活，可定制复杂 Pipeline |
| **专有数据集成** | 标准化数据为主 | 可灵活接入内部数据源 |
| **分布式处理** | 有限 | 原生支持分布式计算（Spark）|

### 数据混合策略

- 公开代码数据（The Stack）
- Replit 平台的用户代码（专有数据，需脱敏处理）
- 其他非 Hugging Face 来源的数据集

> **关键洞见**：对于代码模型，数据不仅要「大」，更要「相关」。Replit 的用户代码数据是 Ghostwriter 的独特优势——公开数据集中没有这部分。

## 3. 数据预处理

原始数据质量参差不齐，直接训练会引入大量噪声。Replit 实施了多层过滤：

### 隐私与安全过滤

- **匿名化（Anonymization）**：移除代码中硬编码的敏感信息
  - 邮件地址
  - IP 地址
  - Secret Keys / API Keys / 密码

### 代码质量过滤

| 过滤规则 | 目标 | 方法 |
|----------|------|------|
| **自动生成代码** | 排除模板代码、机器生成代码 | 正则表达式 + 启发式规则 |
| **压缩/混淆代码（Minified）** | 排除无可读性的代码 | 行长度检测 |
| **无法编译/解析的代码** | 排除语法错误的代码 | 尝试 Parse 后过滤 |
| **平均行长度** | 过滤异常代码结构 | 统计阈值 |
| **最大行长度** | 排除压缩单行代码 | 长度上限 |
| **字母数字字符比例** | 排除二进制/编码内容 | 字符比例阈值 |

### 一个重要的反直觉发现

> **GitHub 指标不等于数据质量**：Stars、Forks 等 GitHub 社交指标**并不能**有效提升训练数据质量。数据质量过滤应基于代码本身的特征，而非外部指标。

这说明「看起来高质量」（受欢迎的仓库）的代码不一定对模型训练更有价值——更重要的是代码的**结构特征**（可解析、无混淆、无噪声）。

## 4. Tokenizer 训练

### 什么是 Tokenizer？

Tokenizer = **分词算法** + **词汇表（Vocabulary）**

```
原始文本/代码
    ↓ Tokenizer
Token ID 序列（整数列表）
    ↓ 模型处理
    ↓ Tokenizer（逆向）
输出文本/代码
```

### 为什么要训练自定义 Tokenizer？

直接使用 GPT-2 或 LLaMA 等模型的 Tokenizer 存在问题：这些 Tokenizer 主要针对**英文自然语言**优化，对代码的处理效率低下。

| 对比维度 | 通用 Tokenizer | 代码专用 Tokenizer |
|----------|---------------|------------------|
| **词汇表内容** | 以英文单词为主 | 包含代码关键字、符号、标识符 |
| **词汇表大小** | 较大（冗余词汇多）| 更小（针对性更强）|
| **编码效率** | 代码需要更多 Token | 相同代码用更少 Token 表示 |
| **训练速度** | 较慢（序列更长）| 更快（序列更短）|
| **推理速度** | 较慢 | 更快 |

### 自定义 Tokenizer 的优势

- **更小的词汇表**：针对代码域训练，冗余词汇少
- **加速训练与推理**：相同内容编码后序列更短，计算量减少
- **捕获领域相关信息**：代码特有的符号组合、缩进模式等能被正确 Tokenize

> **工程细节**：自定义 Tokenizer 需要反馈到整个数据管道——数据预处理完成后，要用新 Tokenizer 重新编码所有训练数据。这是一个迭代循环，不是单向流程。

## 5. 模型训练：MosaicML & Weights & Biases

### MosaicML：训练基础设施

**为什么选择 MosaicML 而不是直接用云厂商 GPU？**

- **多云 GPU 聚合**：从 AWS、GCP、Azure 等多个云厂商汇聚 GPU 资源，降低单价
- **预配置的 LLM 训练方案**：内置针对 LLM 优化的训练配置，无需从零调参
- **容错管理（Fault-tolerant）**：大规模训练中节点故障是常态，MosaicML 提供自动恢复机制
- **简洁的 CLI 接口**：降低管理 GPU 集群的运维复杂度

### 大规模训练的工程挑战

```
单机训练（几小时完成）
    ↓ 扩展到
多机多卡训练（可能持续数天/数周）
    ↓ 带来新问题
节点故障 → 需要 Checkpoint + 自动恢复
通信瓶颈 → 需要优化分布式通信策略
显存限制 → 需要 Mixed Precision、Gradient Checkpointing 等技术
```

### Weights & Biases：实验追踪

训练过程中记录的关键指标：
- **训练损失（Training Loss）**：模型是否在收敛？
- **学习率曲线**：LR Schedule 是否合理？
- **GPU 利用率**：资源是否被充分使用？
- **梯度范数**：是否出现梯度爆炸/消失？

> **核心价值**：W&B 让多次训练运行的指标可对比，帮助工程师快速判断超参数变化的效果，避免「黑盒」式调参。

## 6. 测试与评估：HumanEval & Hugging Face

### 为什么 LLM 评估特别难？

评估语言模型比评估传统软件系统复杂得多：
- 输出是自然语言/代码，没有唯一正确答案
- 评估本身需要大量计算资源（要跑模型生成结果）
- 自动指标不能完全代替人类判断

### 代码模型的标准评估：HumanEval

**HumanEval** 是 OpenAI 发布的代码评估基准数据集：

```python
# HumanEval 的评估方式
# 给模型一个函数签名 + 文档字符串
def add(a: int, b: int) -> int:
    """返回两个整数之和"""
    # 模型需要生成函数体

# 然后用预定义的测试用例验证生成代码的正确性
assert add(1, 2) == 3
assert add(-1, 1) == 0
```

核心指标：**Pass@k**——生成 k 个候选，至少一个通过所有测试用例的概率

### 评估的多重挑战

| 挑战 | 说明 |
|------|------|
| **多语言测试** | 模型需在数百种编程语言上评估，工作量巨大 |
| **专项任务** | 代码补全、网页代码生成等特定场景需要专门评估集 |
| **未见数据评估** | 必须在训练时未见过的数据上评估，防止泄漏偏差 |
| **指标与实用性的鸿沟** | 在 Benchmark 上得分高的模型，不一定在实际使用中好用 |

### 最后一点最重要

> **「Benchmark 好 ≠ 用户体验好」**：这是代码模型评估的核心难题。一个在 HumanEval 上 Pass@1 达到 70% 的模型，可能在实际补全用户代码时表现让人失望。这促使了**人工评估和用户测试**成为不可或缺的补充手段。

## 7. 部署：FasterTransformer、Triton Server & k8s

### 生产部署的核心目标

代码补全产品对延迟极其敏感——用户在打字，需要实时响应。生产部署需要在三个维度上优化：

```
低延迟（Low Latency）
    ↕
高吞吐（High Throughput）
    ↕
成本效率（Cost Efficiency）
```

### FasterTransformer：推理加速引擎

NVIDIA FasterTransformer 是一个针对 Transformer 推理优化的库：
- 算子融合（Kernel Fusion）：把多个 GPU 算子合并，减少内存读写
- 量化支持：INT8/FP16 混合精度推理
- 针对 NVIDIA GPU 深度优化

### NVIDIA Triton Inference Server：服务框架

Triton 提供生产级的模型服务能力：

| 特性 | 说明 |
|------|------|
| **多实例部署** | 一个 GPU 上运行多个模型实例，提升吞吐量 |
| **多 GPU 支持** | 一个模型跨多个 GPU（模型并行）|
| **动态批处理** | 将多个请求合并成一个 Batch 处理，提升 GPU 利用率 |
| **请求取消** | 用户停止输入时取消未完成的推理请求，降低延迟 |

### Kubernetes：自动扩缩容

- 根据请求量自动增减 GPU 实例
- 处理节点故障，保证服务可用性
- **特殊挑战**：GPU 节点比 CPU 节点更难扩缩容
  - 冷启动时间长（加载模型权重需要时间）
  - 特定区域 GPU 资源紧张时可能无法及时扩容
  - 模型体积越大，启动时间越长，扩容响应越慢

> **实战经验**：GPU 短缺不仅是训练阶段的问题，在特定时区和区域，推理阶段也会遇到 GPU 可用性问题，需要提前规划多区域部署策略。

## 8. 经验教训：数据为王、评估与协作

### 教训一：数据是最难的部分

> **「Data is the most difficult part」** —— Reza Shabani

很多人认为「模型架构」或「训练技巧」是 LLM 开发的核心挑战，但 Replit 的实际经验是：**数据管道的质量决定了一切**。

原因：
- 数据质量的微小提升，往往比模型架构调整带来更显著的效果
- 数据 Bug（泄漏、噪声、格式错误）很难发现，但影响深远
- 构建高质量数据管道需要大量工程投入，且没有现成答案

**解决方案**：建立高质量、可扩展的数据管道，支持快速迭代
- 数据处理流程要可重现（Reproducible）
- 支持快速添加新的过滤规则和数据源
- 做好数据版本管理

---

### 教训二：评估需要人工参与

自动化 Benchmark（如 HumanEval）是必要的，但不充分：

```
自动评估（HumanEval 等）
    → 快速、可量化、可对比
    → 但不能完全反映用户体验

人工评估
    → 评估真实使用场景下的实用性
    → 发现 Benchmark 无法暴露的问题

用户测试
    → 最终的真实反馈来自实际用户
    → A/B 测试，测量用户接受率、满意度
```

---

### 教训三：跨团队协作至关重要

训练一个可用的 LLM 产品需要多个职能紧密配合：

| 职能 | 负责内容 |
|------|----------|
| **数据工程** | 数据采集、清洗、管道维护 |
| **ML 研究** | 模型架构、训练策略、超参数调优 |
| **ML 工程** | 训练基础设施、分布式训练 |
| **推理工程** | 部署优化、服务稳定性 |
| **产品/UX** | 用户体验设计、评估指标定义 |

> 各团队相互隔离会导致「模型很好但产品很差」或「产品体验好但模型质量跟不上」的脱节问题。

## 9. 什么是优秀的 LLM 工程师？

Reza 对 LLM 工程师素质的定义，反映了这个角色的特殊性——它介于研究员和软件工程师之间。

### 核心能力

**研究视角（Research Perspective）**：
- 理解模型原理，能读懂论文并判断新技术的适用性
- 会设计实验，能从实验结果中得出有效结论
- 对数据有直觉，知道什么数据会让模型变好/变差

**工程视角（Engineering Perspective）**：
- 大规模分布式数据管道的设计与维护
- PyTorch 及相关训练框架的熟练使用
- CI/CD 实践：自动化测试、模型版本管理、快速迭代

**基础知识**：
- 统计学：理解数据分布、评估指标的统计显著性
- 计算机科学基础：算法、数据结构（处理大规模数据时至关重要）

### CI/CD 在模型训练中的价值

CI/CD（持续集成/持续交付）不只是软件开发的概念，在 LLM 训练中同样关键：

```
代码/数据变更
    ↓ 自动触发
小规模训练验证（快速验证变更是否有效）
    ↓ 通过后
完整训练运行
    ↓ 自动评估
与历史基线对比
    ↓ 通过后
部署到生产
```

这套流程**大幅缩短了迭代周期**，让工程师可以在几小时而非几天内验证一个想法。

---

## 总结：核心认知

| 主题 | 关键认知 |
|------|----------|
| **自训练的价值** | 定制化、隐私、成本、控制权——在特定场景下比 API 更优 |
| **技术栈选择** | 专业工具组合（Databricks + MosaicML + Triton）优于一体化平台 |
| **数据为王** | 数据管道质量 > 模型架构调整，是最难也是最重要的部分 |
| **自定义 Tokenizer** | 领域专用 Tokenizer 带来更高效的编码，加速训练和推理 |
| **评估的局限** | Benchmark 高分 ≠ 用户体验好，人工评估和 A/B 测试不可省略 |
| **部署的复杂性** | GPU 推理部署远比 CPU 服务复杂，批处理和多实例是关键优化手段 |
| **工程师素质** | LLM 工程师需兼具研究视角（理解模型）和工程视角（构建系统）|
| **CI/CD 的价值** | 自动化训练验证流程是加速迭代的关键工程实践 |